In [7]:
import csv
import os
import random
import requests

def download_isic_images(diagnosis, limit, save_dir, records, pool_multiplier=3):
    os.makedirs(save_dir, exist_ok=True)
    url = "https://api.isic-archive.com/api/v2/images/search/"
    params = {
        "query": f'diagnosis_1:"{diagnosis}" AND image_type:"dermoscopic"',
        "limit": limit * pool_multiplier
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json().get("results", [])

    # Slumpa ordningen och plocka ut önskat antal per klass
    random.shuffle(results)
    selected_results = results[:limit]

    for i, item in enumerate(selected_results, 1):
        isic_id = item["isic_id"]
        img_url = item["files"]["full"]["url"]

        img_data = requests.get(img_url)
        img_data.raise_for_status()

        filename = f"{isic_id}_{diagnosis.lower()}_dermoscopic.jpg"
        file_path = os.path.join(save_dir, filename)

        with open(file_path, "wb") as f:
            f.write(img_data.content)

        records.append({
            "filename": filename,
            "file_path": file_path,
            "diagnosis": diagnosis,
            "image_type": "dermoscopic"
        })
        print(f"[{i}/{len(selected_results)}] Laddade ner {file_path}")

all_records = []

download_isic_images("Benign", 10, "data/benign", all_records)
download_isic_images("Malignant", 10, "data/malignant", all_records)

# Valfritt: blanda även radordningen i CSV-filen mellan klasserna
random.shuffle(all_records)

csv_path = "data/labels.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["filename", "file_path", "diagnosis", "image_type"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_records)

print(f"Sparade CSV-fil med {len(all_records)} rader till {os.path.abspath(csv_path)}")

[1/10] Laddade ner data/benign\ISIC_0000003_benign_dermoscopic.jpg
[2/10] Laddade ner data/benign\ISIC_0000024_benign_dermoscopic.jpg
[3/10] Laddade ner data/benign\ISIC_0000010_benign_dermoscopic.jpg
[4/10] Laddade ner data/benign\ISIC_0000039_benign_dermoscopic.jpg
[5/10] Laddade ner data/benign\ISIC_0000019_benign_dermoscopic.jpg
[6/10] Laddade ner data/benign\ISIC_0000007_benign_dermoscopic.jpg
[7/10] Laddade ner data/benign\ISIC_0000000_benign_dermoscopic.jpg
[8/10] Laddade ner data/benign\ISIC_0000028_benign_dermoscopic.jpg
[9/10] Laddade ner data/benign\ISIC_0000037_benign_dermoscopic.jpg
[10/10] Laddade ner data/benign\ISIC_0000021_benign_dermoscopic.jpg
[1/10] Laddade ner data/malignant\ISIC_0000035_malignant_dermoscopic.jpg
[2/10] Laddade ner data/malignant\ISIC_0000074_malignant_dermoscopic.jpg
[3/10] Laddade ner data/malignant\ISIC_0000013_malignant_dermoscopic.jpg
[4/10] Laddade ner data/malignant\ISIC_0000040_malignant_dermoscopic.jpg
[5/10] Laddade ner data/malignant\ISI